![UTN Facultad Regional Mendoza](https://raw.githubusercontent.com/javovelez/Modelos-de-Lenguaje/main/img/logo_utn_frm.png)

# Laboratorio n° 1. Parte C: Embeddings preentrenados

**Asignatura:** Modelos de Lenguaje
**Bloque:** 2 — Redes Neuronales Recurrentes

---

## Introducción

Hasta acá construiste **una sola** clase de *embeddings*, y siempre de la misma manera. En la Parte B, la tabla del clasificador de escenarios empezó en valores al azar y se fue acomodando a fuerza de gradiente, con un único objetivo: acertar el escenario de la orden. Los vectores que salieron de ahí no eran el propósito del modelo, eran su **subproducto**.

Eso tiene una consecuencia que conviene decir en voz alta antes de empezar: esa tabla aprendió *lo que hizo falta para esa tarea*, y nada más. Las palabras que no ayudaban a distinguir un escenario de otro se quedaron prácticamente donde las dejó la inicialización, porque nunca les llegó gradiente. Es una representación **específica de la tarea**, y no se puede llevar a ninguna otra.

La clase teórica mostró la alternativa: vectores entrenados sin ninguna etiqueta, prediciendo qué palabras aparecen cerca de cuáles, sobre cantidades de texto que ningún trabajo práctico puede procesar. Este laboratorio los usa. Vas a descargar una tabla de **fastText** para el español —del orden de $10^{11}$ palabras de entrenamiento— y hacer con ella tres cosas: mirar qué tiene adentro, contrastarla contra una tabla aprendida en una tarea, y medir cuánto sirve de verdad cuando la usás para resolver un problema propio.

La comparación es el hilo del laboratorio. Vas a tener dos tablas de la **misma forma**, `15.548 × 300`, sobre el **mismo vocabulario**: una entrenada para predecir cuántas estrellas tiene una reseña, la otra para predecir contexto sobre medio internet. No comparten casi ningún vecino. Entender de dónde sale esa diferencia es entender qué es lo que un *embedding* representa.

Al completar este laboratorio vas a poder:

- Medir similitud entre palabras por coseno sobre una tabla preentrenada, y evaluar qué parte de tu vocabulario cubre.
- Resolver analogías con aritmética de vectores y explicar qué propiedad del espacio las hace posibles.
- Contrastar una geometría aprendida en una tarea contra una de propósito general, y atribuir la diferencia a la señal de entrenamiento y al corpus por separado.
- Interpretar una proyección a dos dimensiones sabiendo cuánta información conserva.
- Medir el aporte de la transferencia según cuántos datos etiquetados tenga la tarea.
- Identificar tres límites de asignar un vector fijo a cada palabra, y decir qué los causa.

---

## Instrucciones generales

- Completá el código en las celdas marcadas con `# Tu código aquí`.
- Respondé las preguntas de análisis en las celdas de texto (tipo Markdown).
- **Este laboratorio va después de la clase de introducción a los *embeddings***, y da por visto lo que se explicó ahí: la hipótesis distribucional, el *skip-gram*, el muestreo negativo y de dónde salen los vectores que acá se descargan. Acá no se entrena ninguna tabla desde cero.
- De la Unidad 1 se usan la similitud coseno y la proyección con PCA de la Clase 6, y el ciclo de entrenamiento de la Clase 4. Las consignas te mandan a la sección correspondiente cuando hace falta.
- **Este laboratorio corre entero en CPU**, en unos tres minutos contando las descargas de la preparación —el archivo de vectores preentrenados pesa 81 MB—. El cómputo en sí es menos de minuto y medio, y lo más largo es el experimento de transferencia del Ejercicio 5, que entrena seis clasificadores. Si tenés GPU disponible podés usarla, pero no hace falta.
- Las celdas de setup dejan listos el tokenizador y la clase `Vocabulario` con los que se codifica el corpus. Usalos sin modificarlos: si los reemplazás por otra implementación, los números no van a coincidir con los esperados.
- **Fijá las semillas que te pide cada enunciado.** Sin semilla, ninguna de las cifras que imprimas va a ser comparable con nada.

## IMPORTANTE: qué celdas podés modificar

Este laboratorio es un **entregable**. Solo debés completar las celdas de actividad, que son las que aparecen con el comentario `# Tu código aquí` o el texto `*(Escribí tu respuesta acá)*`. Todas las demás celdas —enunciados, explicaciones, ejemplos provistos y encabezado— **no se tocan**.

La corrección se hace celda por celda: cada respuesta se busca en la celda donde el enunciado la pide. Si escribís en otro lado, o si movés, renombrás o borrás celdas del enunciado, esa parte de tu entrega queda sin poder corregirse.

Si querés probar algo suelto, hacelo en la misma celda de actividad o en una celda nueva que agregues, y borrala antes de entregar.

---
## Preparación

Las tres celdas que siguen ya vienen resueltas. No hay nada que completar en ellas, pero **hay que ejecutarlas** en orden antes de empezar, y conviene leer las dos últimas porque definen los nombres que usan todos los ejercicios.

La primera importa las librerías. La segunda descarga el corpus de reseñas. La tercera arma el vocabulario y fija la dimensión de los vectores.

In [ ]:
# ─── Setup: imports ─────────────────────────────────────────────────────────
import os
import time
import urllib.request
import collections

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
import torch.nn.functional as F

print(f"Versión de PyTorch: {torch.__version__}")

### El corpus: reseñas de Amazon en español

Son 200.000 reseñas de productos etiquetadas con la cantidad de estrellas que puso quien las escribió, de 1 a 5 y perfectamente balanceadas. Cada reseña trae el título y el cuerpo separados por un salto de línea doble.

Es el **mismo** corpus que usa el notebook de teoría de la Unidad 1, y no es casualidad: sobre él se entrena el clasificador de estrellas cuya tabla de *embeddings* vas a comparar contra la preentrenada, y sobre él se mide la transferencia del Ejercicio 5.

La celda descarga el corpus —un solo archivo de 16 MB, unos segundos— y lo separa en `train`, `val` y `test`. Después imprime un resumen de largos y una reseña de ejemplo.

Lo baja del repositorio de la materia y no del Hub de Hugging Face, aunque el corpus venga de ahí. El motivo es práctico: cuando el curso entero abre el notebook a la vez desde la misma red, el Hub ve muchas descargas del mismo origen en pocos minutos y empieza a rechazarlas, y la clase se cae en esta celda antes del primer ejercicio. La copia del repositorio no tiene ese límite. El Hub queda igual como respaldo, y la celda lo usa sola si el repositorio no responde.

In [ ]:
# ─── Setup: el corpus de reseñas de Amazon en español ───────────────────────
# Un solo archivo con las tres particiones y una columna `split` que las
# distingue. Sale del repositorio de la materia; el Hub queda de respaldo.
URL_CORPUS = ("https://github.com/javovelez/Modelos-de-Lenguaje/releases/download"
              "/corpus-v1/amazon_reviews_es.parquet")
HUB = ("https://huggingface.co/datasets/mteb/AmazonReviewsClassification"
       "/resolve/main/es")
ARCHIVO_CORPUS = "amazon_reviews_es.parquet"


def leer_corpus():
    """Lee el corpus del repositorio de la materia; si no responde, del Hub."""
    try:
        if not os.path.exists(ARCHIVO_CORPUS):
            urllib.request.urlretrieve(URL_CORPUS, ARCHIVO_CORPUS)
        return pd.read_parquet(ARCHIVO_CORPUS)
    except Exception as falla:
        print(f"el repositorio de la materia no respondió ({falla}).")
        print("voy al Hub de Hugging Face, que es el respaldo.")
        partes = []
        for particion in ["train", "validation", "test"]:
            parte = pd.read_parquet(f"{HUB}/{particion}-00000-of-00001.parquet")
            parte["split"] = particion
            partes.append(parte)
        return pd.concat(partes, ignore_index=True)


corpus_completo = leer_corpus()
train = corpus_completo[corpus_completo.split == "train"].reset_index(drop=True)
val   = corpus_completo[corpus_completo.split == "validation"].reset_index(drop=True)
test  = corpus_completo[corpus_completo.split == "test"].reset_index(drop=True)

print(f"entrenamiento: {len(train):>7,} reseñas")
print(f"validación:    {len(val):>7,} reseñas")
print(f"prueba:        {len(test):>7,} reseñas")
print(f"estrellas:     {sorted(int(e) + 1 for e in train.label.unique())}  (label + 1)")
print()

largos = train.text.str.split().str.len()
print(f"palabras por reseña: media {largos.mean():.1f}, mediana {largos.median():.0f}, "
      f"percentil 95 {np.percentile(largos, 95):.0f}, máximo {largos.max()}")
print()
print("una reseña de 1 estrella:")
print(" ", repr(train.text.iloc[0]))

### El tokenizador, el vocabulario y la dimensión

Esta celda deja disponibles el tokenizador `tok_simple` y la clase `Vocabulario`, y con ellos construye `vocab` sobre el corpus de entrenamiento. Vienen provistos para que los números de este laboratorio sean comparables entre todos. Deja además dos constantes que vas a ver en todos los ejercicios: `V`, el tamaño del vocabulario, y `DIM`, la dimensión de los vectores.

Hay dos decisiones tomadas acá que conviene que entiendas antes de seguir:

- **`freq_min=10`**, mucho más alto que el `freq_min=2` de la Parte A. Para que la fila de una palabra signifique algo hace falta que la palabra aparezca muchas veces. Con diez apariciones ya es discutible; con dos es imposible, y esas filas terminarían siendo ruido con el que después vamos a medir similitudes.
- **`DIM = 300`**, que no es un número elegido por nosotros: es la dimensión de los vectores de fastText que se descargan más adelante. Fijarla acá obliga a que la tabla del clasificador de estrellas tenga exactamente la misma forma que la preentrenada. Sin eso, la comparación del Ejercicio 3 mediría también la diferencia de capacidad entre las dos tablas, y no queremos que mida eso.

In [ ]:
# ─── Setup: el tokenizador, el vocabulario y la dimensión ────────────────────
# El tokenizador y la clase Vocabulario vienen en un módulo auxiliar de la
# materia, que bajamos acá.
URL_PIPELINE = ("https://github.com/javovelez/Modelos-de-Lenguaje/releases/download"
                "/utils-v1/lab1a_pipeline.py")

if not os.path.exists("lab1a_pipeline.py"):
    urllib.request.urlretrieve(URL_PIPELINE, "lab1a_pipeline.py")

from lab1a_pipeline import tok_simple, Vocabulario


vocab = Vocabulario(train.text, tokenizador=tok_simple, freq_min=10)
V = len(vocab)

# La dimensión de los vectores. Es la de fastText, y la tabla del clasificador
# de estrellas la copia para que la comparación del Ejercicio 3 no mida
# capacidad en vez de geometría.
DIM = 300

en_vocab = sum(f for p, f in vocab.contador.items() if p in vocab.stoi)
totales = sum(vocab.contador.values())

print(vocab)
print(f"formas distintas en el corpus: {len(vocab.contador):,}")
print(f"formas en el vocabulario:      {V:,}  (incluye <pad> y <unk>)")
print(f"dimensión de los embeddings:   {DIM}")
print(f"tokens totales:                {totales:,}")
print(f"cobertura de apariciones:      {en_vocab / totales:.2%}")
print()
print("las 15 más frecuentes:", vocab.itos[2:17])

---
## Sección A: Qué hay adentro de una tabla preentrenada

Antes de comparar nada, conviene mirar el objeto con el que vamos a trabajar. Una tabla de *embeddings* preentrenada es un archivo: una lista de palabras y una matriz de números, sin ningún modelo alrededor. Todo lo que se puede hacer con ella se hace con producto punto.

Los dos ejercicios de esta sección son las dos operaciones básicas sobre ese archivo —buscar los vecinos de una palabra y resolver una analogía—, y en los dos casos el ejercicio no termina en el resultado lindo: termina en lo que el resultado no dice.

### Provisto: los vectores preentrenados

La celda que sigue no hay que completarla: descarga vectores de **fastText** para el español, entrenados sobre Common Crawl y Wikipedia. Son del orden de $10^{11}$ palabras de entrenamiento, contra los 6 millones de nuestro corpus de reseñas: cinco órdenes de magnitud más.

El archivo original pesa 1,3 GB y trae 2.000.000 de tokens, así que usamos un recorte: los 150.000 más frecuentes en minúscula, guardados en `float16`. Son 81 MB y se descargan en unos segundos; la precisión que se pierde al usar `float16` no afecta nada de lo que vamos a medir.

Al terminar vas a tener `pre_itos` y `pre_stoi` —el mapeo entre token e índice de esta tabla— y la matriz `E_pre`, ya normalizada fila por fila para que el producto punto entre dos filas sea directamente el coseno.

Prestá atención a un detalle que va a importar en los ejercicios que siguen: **`pre_stoi` no es `vocab.stoi`**. Son dos vocabularios distintos, construidos sobre corpus distintos, y el índice de una palabra en uno no tiene nada que ver con su índice en el otro. Cada tabla se consulta con su propio mapeo.

In [ ]:
# ─── Provisto: descarga de los vectores preentrenados ───────────────────────
URL_VECTORES = ("https://github.com/javovelez/Modelos-de-Lenguaje/releases/download"
                "/vectores-v1/vectores_es_150k.npz")
ARCHIVO = "vectores_es_150k.npz"

if not os.path.exists(ARCHIVO):
    print("bajando los vectores preentrenados...")
    urllib.request.urlretrieve(URL_VECTORES, ARCHIVO)

datos = np.load(ARCHIVO, allow_pickle=True)
pre_itos = list(datos["tokens"])
pre_stoi = {t: i for i, t in enumerate(pre_itos)}

# Los pasamos a float32 y los normalizamos una vez: todo lo que sigue es coseno.
E_pre = torch.tensor(datos["vectors"].astype(np.float32))
E_pre = E_pre / E_pre.norm(dim=1, keepdim=True).clamp(min=1e-8)

print(f"{len(pre_itos):,} tokens, dimensión {E_pre.shape[1]}")
print(f"los 10 más frecuentes: {pre_itos[:10]}")

### Ejercicio 1 — Vecinos por coseno, y qué parte de tu vocabulario está

**Objetivo:** Consultar una tabla preentrenada por similitud coseno y medir cuánto de tu propio corpus queda cubierto por ella.

**Enunciado:**

1. **Escribí la función `vecinos_pre(palabra, k=8)`**, que devuelva las `k` palabras más cercanas a `palabra` por coseno sobre `E_pre`, excluyendo a la palabra consultada. Si la palabra no está en `pre_stoi`, que lo avise en vez de fallar. Imprimí los vecinos de `rey`, `excelente`, `pésimo`, `caro`, `batería` y `cargador`.

2. **Medí la cobertura del vocabulario del corpus.** Imprimí qué proporción de las **formas** de `vocab` está en el archivo preentrenado, qué proporción de las **apariciones** del corpus queda cubierta, cuántas formas faltan, y las 15 formas faltantes más frecuentes.

> **Nota:** Los vecinos por coseno son los de *6.2 Vecinos más cercanos*, de la Clase 6 de la Unidad 1. Lo único distinto acá es la tabla que se consulta: `E_pre` con `pre_stoi` y `pre_itos`, en vez de la del modelo con `vocab`.

> **Pista 1:** Como `E_pre` ya viene normalizada fila por fila, `E_pre @ E_pre[i]` es directamente el vector de cosenos de la palabra `i` contra todas las demás. Para sacar del resultado a la palabra consultada, poné su similitud en −1 antes del `topk`.

> **Pista 2:** La cobertura de apariciones se pondera con `vocab.contador`: no es lo mismo que falte una palabra que aparece 3 veces que una que aparece 3.000. Las dos cifras se calculan sobre `vocab.itos[2:]`, que deja afuera a `<pad>` y `<unk>`.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

a) Entre los ocho vecinos de `caro` hay tres que significan exactamente lo **contrario**. Explicá por qué el procedimiento con el que se entrenaron estos vectores produce ese resultado, y qué es lo que la cercanía en este espacio mide realmente, si no es "significar lo mismo".

b) La cobertura de formas y la de apariciones dan muy distinto. Explicá cuál de las dos es la cifra que importa para decidir si esta tabla sirve para nuestro corpus, y qué haría distinto un modelo basado en **subpalabras** con una forma que no está en la tabla.

*(Escribí tu respuesta acá)*

### Ejercicio 2 — Analogías, y la letra chica

**Objetivo:** Resolver analogías con aritmética de vectores, y medir cuánto del resultado lo aporta el espacio y cuánto la implementación.

**Enunciado:**

Una analogía se plantea como *"a es a b como c es a ?"*, y se resuelve buscando la palabra más cercana por coseno al vector $\mathbf{b} - \mathbf{a} + \mathbf{c}$.

1. **Escribí la función `analogia(a, b, c, k=3, excluir=True)`**, que devuelva las `k` mejores respuestas con su coseno. Cuando `excluir` sea `True` —el valor por defecto—, las tres palabras de la consulta no pueden aparecer en el resultado; cuando sea `False`, se las deja competir. Si alguna de las tres no está en `pre_stoi`, que lo avise en vez de fallar.

2. **Probala** con `("hombre", "rey", "mujer")`, `("madrid", "españa", "parís")`, `("bueno", "mejor", "malo")` y `("comer", "comí", "beber")`, e imprimí las tres mejores respuestas de cada una con su coseno.

3. **Repetí las cuatro con `excluir=False`** e imprimí las dos salidas de manera que se puedan leer una al lado de la otra. La pregunta de análisis se apoya en la diferencia.

> **Pista 1:** Normalizá el vector $\mathbf{b} - \mathbf{a} + \mathbf{c}$ antes de multiplicarlo por `E_pre`, o los números que imprimas no van a ser cosenos.

> **Pista 2:** Para excluir palabras del resultado, poné su similitud en −1 antes del `topk`, igual que hiciste con la palabra consultada en el Ejercicio 1.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

a) La analogía se resuelve buscando el vecino de $\mathbf{b} - \mathbf{a} + \mathbf{c}$. Explicá qué propiedad tiene que tener el espacio de vectores para que esa cuenta dé algo sensato, y por qué esa propiedad no puede aparecer en una tabla entrenada sobre un corpus del tamaño del nuestro.

b) Compará las dos salidas. Decí qué tienen en común las palabras que ganan el primer puesto cuando no se excluye nada, por qué era previsible, y qué te obliga a corregir eso sobre la afirmación *"los embeddings resuelven analogías"*.

*(Escribí tu respuesta acá)*

---
## Sección B: La tabla de la tarea contra la de propósito general

Acá está el experimento que da sentido al laboratorio. Vas a tener dos tablas de *embeddings* de la misma forma, `15.548 × 300`, indexadas por el mismo vocabulario:

| tabla | qué aprendió a predecir | con cuánto texto |
|---|---|---|
| `E_sup` (clasificador) | cuántas estrellas tiene la reseña | 6 millones de palabras de reseñas |
| `E_pre` (fastText) | qué palabras aparecen cerca de cuáles | del orden de $10^{11}$ palabras |

Las dos son representaciones distribuidas, las dos funcionan, y no se parecen en nada. El Ejercicio 3 mide la diferencia y después averigua de dónde sale; el Ejercicio 4 la mira en un gráfico y pregunta cuánto hay que creerle.

Atribuirla es el trabajo difícil, porque las dos tablas difieren en **dos** cosas a la vez: la señal con la que se entrenaron y el corpus sobre el que se entrenaron. La consigna no lo esconde; separar una causa de la otra es justamente lo que se pregunta.

### Provisto: la tabla aprendida en una tarea

La celda que sigue no hay que completarla: entrena sobre este corpus el clasificador de la Parte B, con la misma arquitectura —tabla de *embeddings*, promedio enmascarado, una capa oculta y una de salida— y dos ajustes que impone la tarea. La capa de salida tiene 5 clases en vez de 18, porque acá se predicen estrellas y no escenarios; y la tabla usa `DIM = 300`, la dimensión de los vectores preentrenados, para que las dos tablas tengan exactamente la misma forma.

Además de la clase `ClasificadorEstrellas`, la celda deja definidos los nombres que reaparecen en los ejercicios 3 y 5: `L`, el largo al que se recortan las reseñas; los tensores `X_ent`, `y_ent`, `X_test` e `y_test`; la función `accuracy(m, X, y)`, que evalúa un modelo por lotes; y `E_sup`, la tabla de *embeddings* entrenada y normalizada fila por fila para poder medir cosenos.

Tarda alrededor de medio minuto. Corré esta celda antes de seguir.

In [ ]:
# ─── Provisto: la tabla de embeddings aprendida en una tarea ────────────────
L = 48                                   # el percentil 95 del corpus es 76


class ClasificadorEstrellas(nn.Module):
    """El modelo de la Parte B, con 5 clases en vez de 18."""

    def __init__(self, n_vocab, dim_emb=DIM, dim_oculta=128, n_clases=5, pad_id=0):
        super().__init__()
        self.pad_id = pad_id
        self.embedding = nn.Embedding(n_vocab, dim_emb, padding_idx=pad_id)
        self.oculta = nn.Linear(dim_emb, dim_oculta)
        self.salida = nn.Linear(dim_oculta, n_clases)

    def promediar(self, x):
        vectores = self.embedding(x)
        mascara = (x != self.pad_id).unsqueeze(-1).float()
        return (vectores * mascara).sum(dim=1) / mascara.sum(dim=1).clamp(min=1)

    def forward(self, x):
        return self.salida(F.relu(self.oculta(self.promediar(x))))


def accuracy(m, X, y, lote=2048):
    """Proporción de aciertos de `m` sobre (X, y), evaluando por lotes."""
    m.eval()
    with torch.no_grad():
        return torch.cat([(m(X[i:i + lote]).argmax(1) == y[i:i + lote]).float()
                          for i in range(0, len(X), lote)]).mean().item()


X_ent = vocab.codificar_lote(train.text, L)
y_ent = torch.tensor(train.label.values)
X_test = vocab.codificar_lote(test.text, L)
y_test = torch.tensor(test.label.values)

torch.manual_seed(0)
sup = ClasificadorEstrellas(V, pad_id=vocab.pad_id)
opt_s = torch.optim.Adam(sup.parameters(), lr=1e-3)
crit_s = nn.CrossEntropyLoss()

t0 = time.time()
for epoca in range(4):
    perm = torch.randperm(len(X_ent))
    for i in range(0, len(X_ent), 256):
        idx = perm[i:i + 256]
        perdida = crit_s(sup(X_ent[idx]), y_ent[idx])
        opt_s.zero_grad()
        perdida.backward()
        opt_s.step()

acc = accuracy(sup, X_test, y_test)

E_sup = sup.embedding.weight.detach()
E_sup = E_sup / E_sup.norm(dim=1, keepdim=True).clamp(min=1e-8)

print(f"clasificador de estrellas entrenado en {time.time() - t0:.0f} s")
print(f"accuracy en prueba: {acc:.1%}   (azar con 5 clases: 20,0%)")
print(f"tabla de embeddings: {tuple(sup.embedding.weight.shape)}, "
      f"la misma forma que la preentrenada")

### Ejercicio 3 — Dos geometrías sobre el mismo vocabulario

**Objetivo:** Medir cuánto se parecen las dos tablas y atribuir la diferencia a sus dos causas posibles por separado.

**Enunciado:**

El ejercicio tiene dos partes y cada una va en su propia celda de código.

**Parte A — los vecinos, lado a lado.**

1. **Escribí la función `vecinos_sup(palabra, k=4)`**, la equivalente de `vecinos_pre` pero sobre `E_sup`. Es la misma cuenta con otro mapeo: acá el índice de una palabra es `vocab[palabra]` y el nombre de un índice es `vocab.itos[i]`.

2. Imprimí una comparación con una fila por palabra y dos columnas —los 4 vecinos más cercanos en cada tabla— para: `excelente`, `pésimo`, `caro`, `batería`, `talla`, `libro`, `cargador` y `devolver`. Guardá esa lista en `CONSULTAS`, que se usa en el ítem siguiente.

3. **Cuantificá el desacuerdo.** Para esas mismas ocho palabras, calculá cuántos de sus 10 vecinos más cercanos comparten las dos tablas, y promediá ese conteo sobre las ocho. Imprimí el promedio, que va de 0 a 10.

4. **Compará los cosenos de pares elegidos.** Imprimí una tabla con el coseno en las dos geometrías para `(bueno, malo)`, `(caro, barato)`, `(bien, mal)`, `(excelente, pésimo)` y `(excelente, perfecto)`.

> **Nota:** Las dos tablas están indexadas por vocabularios distintos, así que una sola función no sirve para las dos. `vecinos_pre` consulta `E_pre` con `pre_stoi`/`pre_itos`; `vecinos_sup` consulta `E_sup` con `vocab`. Es la misma operación —normalizar, multiplicar, `topk`— sobre dos mapeos que no tienen nada que ver entre sí.

In [ ]:
# Tu código aquí

**Parte B — por qué la tabla supervisada no tiene vecinos.**

La Parte A deja una rareza sin explicar: en la tabla supervisada los vecinos no significan nada para **ninguna** palabra, ni siquiera para `excelente`, y todos los cosenos quedan pegados a cero. Y sin embargo ese mismo modelo acierta más del 50% de las estrellas. Esta parte averigua cómo conviven las dos cosas.

5. **Recuperá la tabla con la que arrancó el entrenamiento.** La celda de preparación fijó `torch.manual_seed(0)` inmediatamente antes de crear el modelo, así que volver a instanciar un `ClasificadorEstrellas` con esa misma semilla reproduce exactamente la tabla inicial. Normalizala fila por fila y guardala en `E_ini`.

6. **Medí cuánto se movió cada fila**, con el coseno entre su versión inicial y su versión entrenada. Imprimí la media, la mediana y el mínimo sobre todas las formas; las 12 formas que más se movieron, con su frecuencia en el corpus; y el valor que le tocó a cada una de las ocho palabras de `CONSULTAS`.

7. **Comparalo con el ruido.** Sorteá 20.000 pares de filas al azar de la tabla **entrenada**, imprimí la media y el desvío de sus cosenos, y ponelos al lado de $1/\sqrt{\text{DIM}}$, que es el desvío que tendrían vectores al azar en esa cantidad de dimensiones.

8. **Buscá la dirección que sí aprendió.** Construí el vector `polaridad` como la diferencia entre el promedio de las filas de ocho palabras de reseñas buenas y el promedio de las filas de ocho de reseñas malas, normalizalo, proyectá sobre él todas las filas del vocabulario e imprimí las 10 formas de proyección más baja y las 10 más alta.

> **Nota:** El punto 8 usa los pesos **sin normalizar**, `sup.embedding.weight`, no `E_sup`. Lo que se busca es un desplazamiento, y normalizar cada fila a norma 1 es justamente la operación que lo borra.

> **Pista:** En 300 dimensiones, dos vectores al azar tienen coseno cercano a 0 con un desvío de aproximadamente $1/\sqrt{300} \approx 0{,}058$. Ese es el número contra el que hay que leer los cosenos de la tabla supervisada.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

a) En la tabla supervisada los vecinos son ruido para **todas** las palabras y los cosenos están todos pegados a cero, mientras que en la preentrenada llegan a 0,80. Apoyándote en los tres números de la Parte B, explicá por qué la tabla supervisada no tiene geometría de coseno, y por qué eso no le impide al clasificador acertar más del 50%.

b) Mirá qué formas encabezan la lista de las que más se movieron, cuánto se movió cada una de las ocho de la Parte A y qué ordena la proyección del último punto. Decí qué es lo único que esta tabla llegó a representar y qué conclusión general sacás sobre qué representa un *embedding*. Al comparar `excelente` contra `talla`, prestá atención a las frecuencias: descartan una explicación posible.

*(Escribí tu respuesta acá)*

### Ejercicio 4 — Las dos nubes, y cuánto hay que creerles

**Objetivo:** Ver la diferencia entre las dos geometrías en un gráfico, y medir cuánta información de la geometría original sobrevive a ese gráfico.

**Enunciado:**

1. **Calculá la estrella promedio de cada palabra**: para cada forma del vocabulario, el promedio de estrellas de las reseñas de entrenamiento en las que aparece. Guardalo en `tono`. Contá una vez por reseña, no una vez por aparición.

2. **Proyectá las dos tablas a dos dimensiones con PCA y graficalas lado a lado**, en una figura de dos paneles, sobre esta lista de palabras:

```python
PALABRAS = ["excelente", "perfecto", "genial", "estupendo", "recomendable", "encantada",
            "pésimo", "horrible", "malísimo", "defectuoso", "decepcionante", "estafa",
            "caro", "barato", "precio", "calidad",
            "batería", "pantalla", "cargador", "talla", "camiseta", "libro",
            "cocina", "sonido", "envío", "tamaño"]
```

3. **Coloreá cada punto por su estrella promedio**, con el mapa `RdYlGn`. Anotá cada punto con su palabra, y poné título, grilla y barra de color en los dos paneles.

4. **Calculá y mostrá en cada título qué proporción de la varianza conservan los dos componentes** que estás graficando. Es la suma de los cuadrados de los dos primeros valores singulares dividida por la suma de cuadrados de toda la submatriz centrada.

> **Nota:** La lista `PALABRAS` tiene que quedar filtrada a las formas que están en **las dos** tablas: `vocab.stoi` y `pre_stoi` son vocabularios distintos y una palabra puede faltar en uno de los dos.

> **Pista 1:** El PCA y el coloreado por estrellas son los de *6.3 La geometría del espacio aprendido*, de la Clase 6. Ahí está `torch.pca_lowrank`, con el detalle de que la submatriz hay que centrarla antes de proyectar. Lo nuevo acá es hacerlo dos veces y quedarse también con los valores singulares.

> **Pista 2:** Fijá `vmin=2.0, vmax=4.0` en el `scatter` para que la escala de color sea la misma en los dos paneles. Sin eso la comparación visual no vale.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

a) Los dos paneles están organizados, pero no por lo mismo. Describí qué separa cada uno, y explicá por qué el criterio de cada panel es el que corresponde a la señal con la que se entrenó esa tabla.

b) Los dos componentes que graficaste conservan una fracción baja de la varianza. Decí qué conclusiones de las que sacaste en a) siguen en pie con ese dato a la vista y cuáles no, y qué error de lectura habilita mostrar una proyección sin informar esa cifra.

*(Escribí tu respuesta acá)*

---
## Sección C: Cuánto valen

Las dos secciones anteriores describieron las tablas. Esta las pone a trabajar y las mide.

El Ejercicio 5 vuelve al clasificador de estrellas y le cambia una sola cosa: de dónde salen los valores iniciales de su tabla de *embeddings*. Con eso se contesta la pregunta práctica —¿conviene usar vectores preentrenados?— y se descubre que la respuesta no es una sola, porque depende de cuántos datos etiquetados tenga la tarea.

El Ejercicio 6 cierra el laboratorio con los límites de todo lo que viste acá, que son los que motivan la unidad que sigue.

### Ejercicio 5 — Transferencia: ¿cuánto sirven de verdad?

**Objetivo:** Medir qué aportan los vectores preentrenados como inicialización de una tarea propia, y descubrir que la respuesta depende de cuántos datos etiquetados tengas.

**Enunciado:**

Vas a reusar la clase `ClasificadorEstrellas` de la celda de preparación y comparar **tres maneras de inicializar su tabla de *embeddings***, en dos regímenes de datos. El ejercicio tiene dos partes y cada una va en su propia celda de código.

**Parte A — lo que hace falta antes de medir.**

1. **Construí los tensores de validación** `X_val` e `y_val` a partir de `val`, igual que la celda de preparación construyó los de entrenamiento y prueba, y con el mismo `L`.

2. **Construí la matriz de inicialización** `M_pre`, de forma `(V, DIM)`: para cada forma de `vocab.itos`, su vector preentrenado si existe; para las que no existen, un vector al azar `normal(0, sigma)` con `sigma` igual al desvío estándar de `E_pre`. La fila de `<pad>` va en ceros. Usá `np.random.default_rng(0)` e imprimí cuántas filas quedaron preentrenadas y cuántas al azar.

3. **Barajá `X_ent`/`y_ent` antes de recortar**, con `torch.randperm` y semilla 0, y guardá el resultado en `X_baraja`/`y_baraja`. El corpus viene ordenado por estrellas: quedarte con las primeras 2.000 filas sin barajar te daría una sola clase.

In [ ]:
# Tu código aquí

**Parte B — el experimento.**

4. **Escribí la función `correr(nombre, pesos, congelar, n, epocas)`**, que entrene un `ClasificadorEstrellas` sobre las primeras `n` reseñas de `X_baraja` y devuelva la accuracy de prueba **en la época de mejor accuracy de validación**. La época se elige mirando validación y nunca prueba: elegirla por prueba contamina la estimación con la partición que después se reporta, y para eso justamente hay una partición de validación. Tenés `accuracy(m, X, y)` lista desde la celda de preparación. Las tres configuraciones son:

   - `pesos=None` → tabla al azar, entrenable.
   - `pesos=M_pre, congelar=True` → preentrenada, `requires_grad = False`.
   - `pesos=M_pre, congelar=False` → preentrenada, ajuste fino.

   Fijá `torch.manual_seed(0)` al empezar cada corrida, y pasale al optimizador solo los parámetros con `requires_grad`.

5. **Corré las tres configuraciones con `n = 2.000` (20 épocas) y con `n = 200.000` (4 épocas)**, e imprimí una tabla con las seis accuracies. Agregá una columna con la cantidad de parámetros entrenables de cada configuración.

> **Nota:** Cargar una matriz de pesos en una `nn.Embedding` ya creada y congelarla no apareció en la Unidad 1, así que va explicado: los pesos se copian sobre `m.embedding.weight.data`, y para que dejen de recibir gradiente se pone `m.embedding.weight.requires_grad = False`. Las dos cosas se hacen sobre el mismo atributo `.weight`.

> **Pista:** Con la tabla congelada solo se entrenan las dos capas lineales, unas decenas de miles de parámetros contra los millones de la tabla. Esa diferencia es parte de lo que explica el resultado.

In [ ]:
# Tu código aquí

**Pregunta de análisis:**

a) La ventaja de inicializar con vectores preentrenados es mucho más grande con 2.000 reseñas que con 200.000. Explicá por qué se achica, y qué regla general de la transferencia se sigue de eso.

b) La tabla congelada **cambia de lado** entre los dos regímenes: con 2.000 reseñas le gana a la inicialización al azar y con 200.000 le pierde. Explicá el cruce usando la cantidad de parámetros entrenables de cada configuración, y decí qué les falta a los vectores de fastText para esta tarea en particular, apoyándote en lo que mediste en el Ejercicio 3.

*(Escribí tu respuesta acá)*

### Provisto: tres límites de estos vectores

La celda que sigue no hay que completarla: es el experimento que da pie al último ejercicio. Muestra tres cosas que estos vectores no pueden hacer —una palabra polisémica tiene un solo vector, las asociaciones del texto de entrenamiento quedan grabadas en la geometría, y el orden de las palabras sigue sin existir— y para cada una imprime la evidencia.

Usa las funciones `vecinos_pre` y `analogia` que escribiste en los ejercicios 1 y 2, así que si todavía no los resolviste va a dar `NameError`. Observá la salida con atención antes de responder.

In [ ]:
# ─── Provisto: tres límites, en tres líneas ─────────────────────────────────
# 1. Una palabra, un vector, aunque la palabra tenga dos sentidos.
print("polisemia")
for p in ["muñeca", "sierra", "banco", "gato"]:
    print(f"  {p:9s} -> {', '.join(vecinos_pre(p, 7))}")

# 2. Lo que estaba en el texto de entrenamiento, queda en los vectores.
#    Atención con el segundo caso: la respuesta morfológicamente correcta existe en
#    la tabla, y aun así el modelo prefiere otra.
print("\nasociaciones aprendidas del corpus")
for a, b, c in [("hombre", "ingeniero", "mujer"),
                ("hombre", "jefe", "mujer"),
                ("él", "médico", "ella")]:
    print(f"  {a} : {b} :: {c} : ?  ->  {analogia(a, b, c, k=4)}")
print(f"  para comparar, dónde quedó 'médica': puesto "
      f"{[w for w, _ in analogia('él', 'médico', 'ella', k=30)].index('médica') + 1}")

# 3. El orden sigue sin existir.
print("\ndos frases, mismas palabras")
f1, f2 = "el envío fue rápido pero el producto es malo", \
         "el envío fue malo pero el producto es rápido"
for f in (f1, f2):
    ids = [pre_stoi[t] for t in tok_simple(f) if t in pre_stoi]
    print(f"  {f!r}\n     promedio de sus vectores: norma {E_pre[ids].mean(0).norm():.4f}")
iguales = torch.allclose(
    E_pre[[pre_stoi[t] for t in tok_simple(f1) if t in pre_stoi]].mean(0),
    E_pre[[pre_stoi[t] for t in tok_simple(f2) if t in pre_stoi]].mean(0))
print(f"  ¿los dos promedios son idénticos? {iguales}")

### Ejercicio 6 — Tres límites de una tabla de vectores

**Objetivo:** Cerrar el laboratorio identificando qué es lo que esta representación no puede hacer, que es lo que motiva todo lo que sigue en la materia.

**Enunciado:**

La celda de arriba muestra tres límites de asignarle **un vector fijo a cada palabra**. Respondé, apoyándote en la salida:

1. **Polisemia.** Las cuatro palabras de la primera lista tienen dos sentidos en español, pero la salida muestra **dos comportamientos distintos**: en dos de ellas los vecinos vienen de los dos sentidos mezclados, y en las otras dos uno de los sentidos directamente no aparece. Identificá cuáles son cuáles, explicá qué le pasa al vector en cada caso, y por qué ninguna cantidad de datos de entrenamiento lo arregla mientras la tabla tenga una sola fila por palabra.

2. **Asociaciones del corpus.** De las tres analogías, la primera devuelve la respuesta esperable y las otras dos no. En el caso de `médico`, notá que la forma femenina correcta **está** en la tabla y aun así queda por debajo. Explicá de dónde sale esa asociación y por qué llamarla "sesgo del modelo" describe mal el problema. Nombrá una consecuencia concreta de usar estos vectores sin más en un sistema que tome decisiones sobre personas.

El tercer límite no hay que analizarlo, porque ya lo viste en la Parte B: las dos frases del final tienen las mismas palabras y significados opuestos, y su promedio de vectores es idéntico. La información no se pierde en los vectores sino **después**, en la suma que los combina, y como la suma es conmutativa da lo mismo qué tan buenos sean los vectores de entrada. Lo que tiene que cambiar es la operación que los junta, y de eso se ocupa la unidad que sigue.

*(Escribí tu respuesta acá)*

---
## Antes de entregar

Revisá esta checklist rápida:

- [ ] Reinicié el entorno y ejecuté **todas** las celdas de arriba a abajo sin errores (**Entorno de ejecución > Reiniciar y ejecutar todo**).
- [ ] Fijé todas las semillas que pedían los enunciados, así que mis números son reproducibles.
- [ ] Las dos salidas del Ejercicio 2 —con exclusión y sin exclusión— están impresas las dos, porque la pregunta se apoya en la diferencia.
- [ ] La tabla del Ejercicio 3 compara las mismas ocho palabras en las dos geometrías, y las consulté con el mapeo que le corresponde a cada tabla.
- [ ] La figura del Ejercicio 4 tiene los dos paneles con la **misma escala de color**, y la proporción de varianza conservada aparece en los dos títulos.
- [ ] La tabla del Ejercicio 5 tiene las seis accuracies y todas están bastante por encima del 20% del azar.
- [ ] Respondí las cinco preguntas de análisis (Ej. 1 a 5) y el análisis final del Ejercicio 6.
- [ ] No modifiqué ninguna celda fuera de las de actividad.

---
## ¡Listo!

Trabajaste con una tabla de *embeddings* que no entrenaste vos: 150.000 palabras en español, 300 dimensiones, y del orden de $10^{11}$ palabras de texto detrás. Le buscaste vecinos, le resolviste analogías, mediste qué parte de tu vocabulario cubre, y la usaste para inicializar un clasificador propio.

Y en el medio quedó el resultado que importa, el del Ejercicio 3. Dos tablas de la misma forma, sobre el mismo vocabulario, y casi sin un vecino en común. Un *embedding* no representa "el significado" de una palabra: representa **lo que hizo falta para resolver la tarea con la que se entrenó**. La preentrenada sabe de qué se habla y confunde `caro` con `barato`; la supervisada no los confunde, pero no porque los distinga: aprendió una sola dirección, la que va de `estafa` a `encantada`, y dejó todo el resto de sus filas donde las había puesto el azar. Ninguna de las dos es la representación correcta, porque no existe tal cosa fuera de una tarea.

El Ejercicio 5 le puso número a la consecuencia práctica: partir de vectores preentrenados vale mucho cuando hay pocos datos etiquetados y casi nada cuando hay muchos.

Todo esto sigue teniendo el mismo techo que la Parte B, y el Ejercicio 6 lo dejó a la vista: una palabra, un vector, y un promedio que borra el orden. *"El envío fue rápido pero el producto es malo"* y *"el envío fue malo pero el producto es rápido"* siguen siendo la misma entrada.

Lo que viene son las dos maneras de romper ese techo: procesar la secuencia arrastrando un estado, y dejar que cada palabra mire a las demás.